# 03 — Ingestion ROME + Wikidata

Ce notebook ingère deux sources et les range au format canonique :

**PARTIE A — ROME** Grâce au jeu de données ROME, on récupère pour chaque métier IT. Concrètement :
- les **savoir-faire** (compétences techniques → *hard skills*),
- les **savoir-être professionnels** (→ *soft skills*),
- les **savoirs** (connaissances → traités comme *hard*),
- et les **appellations** (synonymes de titres de poste).

**PARTIE B — Wikidata.** Obtenir des concepts émergents et des libellés, via une requête SPARQL.

## PARTIE A — ROME Dataset

### A.1 Métiers IT

On sélectionne les codes ROME du domaine **M18** (informatique). Chaque code est un métier.

In [1]:
import os
import csv
import jobkb_common as C
from collections import Counter, defaultdict


ROME = "ROME"
DOMAIN = C.ROME_DOMAIN_IN_SCOPE  # "M18" (informatique)

def load_rome(name):
    path = os.path.join(C.ROME_DIR, name)
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

# noms des fichiers du jeu de données complet (v461)
F_CODES   = "unix_referentiel_code_rome_v461_utf8.csv"
F_APPEL   = "unix_referentiel_appellation_v461_utf8.csv"
F_COMPET  = "unix_referentiel_competence_v461_utf8.csv"
F_SAVOIR  = "unix_referentiel_savoir_v461_utf8.csv"
F_LIENS   = "unix_liens_rome_referentiels_v461_utf8.csv"

print("Fichiers ROME v4.0 attendus dans :", C.ROME_DIR)

Fichiers ROME v4.0 attendus dans : d:\JobKB-final\Datasets\ROME


In [2]:
codes = load_rome(F_CODES)
m18 = [c for c in codes if c["code_rome"].startswith(DOMAIN)]
print(f"Métiers ROME du domaine {DOMAIN} (informatique) : {len(m18)}")
for c in m18[:8]:
    print(f"   {c['code_rome']}  {c['libelle_rome']}")

m18_codes = {c["code_rome"] for c in m18}

Métiers ROME du domaine M18 (informatique) : 96
   M1801  Administrateur / Administratrice de systèmes d'information (SI)
   M1802  Expert / Experte systèmes et réseaux informatiques
   M1803  Directeur / Directrice des systèmes d'information (DSI)
   M1804  Ingénieur / Ingénieure télécoms et environnement
   M1805  Développeur / Développeuse informatique
   M1806  Consultant fonctionnel / Consultante fonctionnelle des systèmes d'information
   M1807  Spécialiste outils, systèmes d'exploitation, réseaux et télécoms
   M1808  Cartographe


### A.2 Synonymes de titres de poste

Le fichier des appellations liste, pour chaque code ROME, les intitulés d'emploi réels. On les utilise comme **synonymes** du métier.

In [3]:
appellations = load_rome(F_APPEL)
appel_by_code = defaultdict(list)
for a in appellations:
    code = a.get("code_rome", "")
    if code in m18_codes:
        lib = (a.get("libelle_appellation_long") or "").strip()
        if lib:
            appel_by_code[code].append(lib)

total_app = sum(len(v) for v in appel_by_code.values())
print(f"Appellations M18 (synonymes potentiels) : {total_app} sur {len(appel_by_code)} métiers")
print("\n")
ex = m18[4]["code_rome"]
print(f"Exemple {ex} :", appel_by_code.get(ex, [])[:5])

Appellations M18 (synonymes potentiels) : 593 sur 96 métiers


Exemple M1805 : ['Analyste développeur / Analyste développeuse', 'Analyste-programmeur / Analyste-programmeuse gestion informatique', 'Analyste-programmeur / Analyste-programmeuse informatique', 'Analyste-programmeur / Analyste-programmeuse scientifique informatique', 'Développeur / Développeuse informatique']


### A.3 Construction des professions ROME

Chaque métier M18 devient une occupation, avec ses synonymes rangés en libellés alternatifs.

In [4]:
rome_occ_rows, rome_label_rows = [], []
occ_eid_by_code = {}

for c in m18:
    code = c["code_rome"]
    pref = c["libelle_rome"].strip()
    eid = C.mint_id("OCC_", ROME, code)
    occ_eid_by_code[code] = eid
    # synonymes = appellations != libellé préféré, dédupliqués
    syns, seen = [], set()
    for s in appel_by_code.get(code, []):
        if s != pref and s.lower() not in seen:
            seen.add(s.lower())
            syns.append(s)
    rome_occ_rows.append({
        "entity_id": eid, "source": ROME, "source_id": code, "isco_code": "",
        "pref_label_fr": pref, "pref_label_en": "",
        "alt_labels_fr": " | ".join(syns), "alt_labels_en": "",
        "description_fr": "", "description_en": "",
        "occupation_type": "rome_fiche", "label_language_status": "fr_native",
    })
    rome_label_rows += C.make_label_rows(eid, "occupation", ROME,
        preferred={"fr": [pref]}, alts={"fr": syns})

n_syn = sum(1 for l in rome_label_rows if l["label_type"] == "alt")
print(f"{len(rome_occ_rows)} professions ROME, {len(rome_label_rows)} libellés ({n_syn} synonymes)")

print("\nMétiers émergents :")
for r in rome_occ_rows:
    if any(k in r["pref_label_fr"].lower() for k in
           ["devops", "cloud", "data engineer", "blockchain", "jumeau",
            "intelligence artificielle", "pentest", "scrum", "product owner", "cyber"]):
        print(f"   {r['source_id']}  {r['pref_label_fr']}")

96 professions ROME, 593 libellés (497 synonymes)

Métiers émergents :
   M1811  Data engineer
   M1814  Scrum Master
   M1819  Technicien / Technicienne en cybersécurité
   M1822  Spécialiste Jumeau Numérique
   M1827  Ingénieur / Ingénieure DevOps
   M1846  Ingénieur / Ingénieure Cybersécurité Datacenter
   M1856  Expert / Experte en cybersécurité
   M1860  Architecte cloud
   M1864  Product Owner
   M1865  Ingénieur / Ingénieure blockchain
   M1866  Pentesteur / Pentesteuse
   M1876  Technicien / Technicienne Cloud
   M1877  Développeur / Développeuse blockchain
   M1879  Ingénieur / Ingénieure Cloud computing
   M1889  Ingénieur / Ingénieure en Intelligence Artificielle (IA)


### A.4 Compétences détaillées — savoir-faire, savoir-être, savoirs

La table `unix_liens_rome_referentiels` relie chaque métier (`code_rome`) à des items (`code_ogr`) par **bloc** et **rubrique** :
- `code_compo_bloc = 5` → bloc **Compétences** ;
- au sein de ce bloc, `code_rubrique` distingue : `1` = **savoir-faire** (hard), `2` = **savoir-être** (soft), `3` = **savoirs** (connaissances, traitées en hard).


In [5]:
# Références de compétences et de savoirs
comp_ref = {r["code_ogr"]: r for r in load_rome(F_COMPET)}          # savoir-faire + savoir-être
savoir_ref = {r["code_ogr_savoir"]: r for r in load_rome(F_SAVOIR)}  # savoirs (connaissances)

# Liens métier -> items, bloc 5 (Compétences), pour les métiers M18
liens = load_rome(F_LIENS)
bloc5 = [l for l in liens if l["code_rome"] in m18_codes and l["code_compo_bloc"] == "5"]
print(f"Liens compétence (bloc 5) pour M18 : {len(bloc5)}")
print("  par rubrique :", dict(Counter(l["code_rubrique"] for l in bloc5)))

# rubrique -> (nature de compétence, méthode)
RUBRIQUE = {
    "1": ("hard", "rome_savoir_faire"),
    "2": ("soft", "rome_savoir_etre"),
    "3": ("hard", "rome_savoir"), 
}

def skill_label(link):
    """Retourne (libellé, sous_categorie) pour un lien bloc-5 selon sa rubrique."""
    rub = link["code_rubrique"]
    ogr = link["code_ogr"]
    if rub in ("1", "2") and ogr in comp_ref:
        r = comp_ref[ogr]
        return r["libelle_competence"].strip(), r.get("sous_cat_comp", "")
    if rub == "3" and ogr in savoir_ref:
        r = savoir_ref[ogr]
        return r["libelle_savoir"].strip(), r.get("categorie_savoir", "")
    return None, None

Liens compétence (bloc 5) pour M18 : 5296
  par rubrique : {'1': 2758, '2': 318, '3': 2220}


In [6]:
# Construction des compétences et des liens
skill_rows_by_key = {}   # (norm_label, nature) -> skill row
rel_rows = []
seen_rel = set()

def skill_id(norm_label, nature):
    # nature dans l'id pour éviter qu'un même libellé hard et soft ne fusionne par erreur
    return C.mint_id("SKL_", ROME, f"{nature}:{norm_label}")

for l in bloc5:
    rub = l["code_rubrique"]
    if rub not in RUBRIQUE:
        continue
    nature, method = RUBRIQUE[rub]
    label, subcat = skill_label(l)
    if not label:
        continue
    norm = C.normalize_label(label)
    if not norm:
        continue
    key = (norm, nature)
    seid = skill_id(norm, nature)
    if key not in skill_rows_by_key:
        skill_rows_by_key[key] = {
            "entity_id": seid, "source": ROME, "source_id": f"{nature}:{norm}",
            "pref_label_fr": label, "pref_label_en": "",
            "alt_labels_fr": "", "alt_labels_en": "",
            "description_fr": "", "description_en": "",
            "esco_skill_type": "", "esco_reuse_level": "",
            "hard_soft_provisional": nature, "hard_soft_method": method,
        }
    # lien métier -> compétence
    oeid = occ_eid_by_code.get(l["code_rome"])
    if oeid:
        rk = (oeid, seid)
        if rk not in seen_rel:
            seen_rel.add(rk)
            rel_rows.append({"occupation_entity_id": oeid, "skill_entity_id": seid,
                             "relation_type": "essential", "source": ROME})

skill_rows = list(skill_rows_by_key.values())
n_hard = sum(1 for s in skill_rows if s["hard_soft_provisional"] == "hard")
n_soft = sum(1 for s in skill_rows if s["hard_soft_provisional"] == "soft")
print(f"Compétences ROME créées : {len(skill_rows)}  ({n_hard} hard, {n_soft} soft)")
print(f"Liens métier→compétence : {len(rel_rows)}")

# libellés pour les compétences
skill_label_rows = []
for s in skill_rows:
    skill_label_rows += C.make_label_rows(s["entity_id"], "skill", ROME,
                                          preferred={"fr": [s["pref_label_fr"]]})

print("\nExemples de savoir-être (soft) ROME :")
for s in skill_rows:
    if s["hard_soft_provisional"] == "soft":
        print("   -", s["pref_label_fr"])

Compétences ROME créées : 2162  (2148 hard, 14 soft)
Liens métier→compétence : 5296

Exemples de savoir-être (soft) ROME :
   - Faire preuve d'autonomie
   - Organiser son travail selon les priorités et les objectifs
   - Faire preuve de rigueur et de précision
   - Etre force de proposition
   - Faire preuve de leadership
   - Avoir l'esprit d'équipe
   - Etre ouvert aux changements
   - Faire preuve de créativité, d'inventivité
   - Faire preuve de persévérance
   - Inspirer, donner du sens
   - Faire preuve de sens des responsabilités
   - Faire preuve de réactivité
   - Faire preuve de curiosité, d'ouverture d'esprit
   - Etre à l'écoute, faire preuve d'empathie


### A.5 Écriture au format canonique

On écrit occupations, compétences, libellés et liens métier→compétence.

In [7]:
C.replace_source_rows(C.OCCUPATIONS_CSV, C.OCCUPATION_FIELDS, ROME, rome_occ_rows)
C.replace_source_rows(C.SKILLS_CSV, C.SKILL_FIELDS, ROME, skill_rows)
C.replace_source_rows(C.LABELS_CSV, C.LABEL_FIELDS, ROME, rome_label_rows + skill_label_rows)
C.replace_source_rows(C.OCC_SKILL_REL_CSV, C.REL_FIELDS, ROME, rel_rows)

C.log_provenance(ROME, [{
    "entity_id": "ALL_ROME_M18", "source": ROME,
    "source_version": "open data v4.0 (v461, UTF-8)",
    "retrieved_at": C.now_iso(),
    "retrieval_method": "referentiel_code_rome + appellation + competence + savoir + liens_rome_referentiels",
    "notes": f"{len(rome_occ_rows)} métiers M18, {n_syn} synonymes, "
             f"{len(skill_rows)} compétences ({n_hard} hard/{n_soft} soft), {len(rel_rows)} liens. "
             f"Jeu complet : savoir-faire + savoir-être + savoirs inclus.",
}])
print("ROME data écrit dans la couche canonique.")
print(f"  {len(rome_occ_rows)} métiers | {len(skill_rows)} compétences | {len(rel_rows)} liens métier→compétence")

ROME data écrit dans la couche canonique.
  96 métiers | 2162 compétences | 5296 liens métier→compétence


## PARTIE B — Wikidata

Stratégie de requête : on récupère les entités qui sont des **professions** (`wdt:P279*` de `Q28640`) **dont le domaine** (`P425`) est — ou dérive de — l'un des concepts d'ancrage informatiques (informatique `Q21198`, langage de programmation `Q9143`, base de données `Q638608`, réseau informatique `Q1301371`, génie logiciel `Q80993`).

In [8]:
WIKIDATA = "WIKIDATA"
ENDPOINT = "https://query.wikidata.org/sparql"

USER_AGENT = "JobKB-research-bot/0.1 (https://github.com/; mailto:votre-email@exemple.fr) python-SPARQLWrapper"

ANCHORS = ["Q21198", "Q9143", "Q638608", "Q1301371", "Q80993"]
anchor_values = " ".join(f"wd:{q}" for q in ANCHORS)

WIKIDATA_QUERY = f"""
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX schema: <http://schema.org/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?item ?labelFr ?labelEn ?descFr ?descEn ?altFr ?altEn WHERE {{
  ?item wdt:P279* wd:Q28640 .
  ?item wdt:P425 ?field .
  ?field wdt:P279* ?anchor .
  VALUES ?anchor {{ {anchor_values} }}
  OPTIONAL {{ ?item rdfs:label ?labelFr . FILTER(LANG(?labelFr) = "fr") }}
  OPTIONAL {{ ?item rdfs:label ?labelEn . FILTER(LANG(?labelEn) = "en") }}
  OPTIONAL {{ ?item schema:description ?descFr . FILTER(LANG(?descFr) = "fr") }}
  OPTIONAL {{ ?item schema:description ?descEn . FILTER(LANG(?descEn) = "en") }}
  OPTIONAL {{ ?item skos:altLabel ?altFr . FILTER(LANG(?altFr) = "fr") }}
  OPTIONAL {{ ?item skos:altLabel ?altEn . FILTER(LANG(?altEn) = "en") }}
  FILTER(BOUND(?labelFr) || BOUND(?labelEn))
}}
LIMIT 3000
"""
print("Requête SPARQL prête (", len(WIKIDATA_QUERY), "caractères ).")

Requête SPARQL prête ( 971 caractères ).


In [9]:
import time, urllib.error

def fetch_wikidata(query, max_retries=5, base_sleep=5):
    """Interroge le service SPARQL de Wikidata avec gestion du débit."""
    from SPARQLWrapper import SPARQLWrapper, JSON
    sparql = SPARQLWrapper(ENDPOINT, agent=USER_AGENT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.setTimeout(180)
    for attempt in range(1, max_retries + 1):
        try:
            return sparql.query().convert()["results"]["bindings"]
        except urllib.error.HTTPError as e:
            if e.code != 429 or attempt == max_retries:
                raise
            ra = e.headers.get("Retry-After") if hasattr(e, "headers") else None
            try:
                wait = int(ra)
            except (TypeError, ValueError):
                wait = base_sleep * (2 ** (attempt - 1))   # backoff exponentiel de secours
            wait = min(wait, 90) + 1
            print(f"  [429] débit limité — tentative {attempt}/{max_retries}, "
                  f"attente {wait}s (Retry-After={ra}) ...")
            time.sleep(wait)
    raise RuntimeError("inatteignable")

# Exécution live avec back-off automatique.
bindings = None
try:
    bindings = fetch_wikidata(WIKIDATA_QUERY)
    print(f"Wikidata a renvoyé {len(bindings)} lignes (une par combinaison item/alias).")
except Exception as e:
    print("[Wikidata NON ingéré]", type(e).__name__, str(e)[:200])
    print("Causes possibles : pas d'accès internet, 'pip install SPARQLWrapper' manquant,")
    print("ou incident/débit du service Wikidata (réessayez dans quelques minutes).")

Wikidata a renvoyé 9 lignes (une par combinaison item/alias).


In [10]:
# On regroupe les libellés préférés et l'ensemble des alias (une entité (QID) peut apparaître sur plusieurs lignes).
if bindings:
    agg = {}
    for b in bindings:
        qid = b["item"]["value"].rsplit("/", 1)[-1]
        a = agg.setdefault(qid, {"label_fr": "", "label_en": "", "desc_fr": "",
                                 "desc_en": "", "alt_fr": set(), "alt_en": set()})
        a["label_fr"] = a["label_fr"] or b.get("labelFr", {}).get("value", "")
        a["label_en"] = a["label_en"] or b.get("labelEn", {}).get("value", "")
        a["desc_fr"]  = a["desc_fr"]  or b.get("descFr", {}).get("value", "")
        a["desc_en"]  = a["desc_en"]  or b.get("descEn", {}).get("value", "")
        if b.get("altFr", {}).get("value"): a["alt_fr"].add(b["altFr"]["value"])
        if b.get("altEn", {}).get("value"): a["alt_en"].add(b["altEn"]["value"])

    wd_occ_rows, wd_label_rows = [], []
    for qid, a in agg.items():
        eid = C.mint_id("OCC_", WIKIDATA, qid)
        status = "fr_native" if a["label_fr"] else "en_only_pending_fr"
        pref = {}
        if a["label_fr"]: pref["fr"] = [a["label_fr"]]
        if a["label_en"]: pref["en"] = [a["label_en"]]
        alts = {}
        if a["alt_fr"]: alts["fr"] = sorted(a["alt_fr"])
        if a["alt_en"]: alts["en"] = sorted(a["alt_en"])
        wd_occ_rows.append({
            "entity_id": eid, "source": WIKIDATA, "source_id": qid, "isco_code": "",
            "pref_label_fr": a["label_fr"], "pref_label_en": a["label_en"],
            "alt_labels_fr": " | ".join(sorted(a["alt_fr"])),
            "alt_labels_en": " | ".join(sorted(a["alt_en"])),
            "description_fr": a["desc_fr"], "description_en": a["desc_en"],
            "occupation_type": "wikidata_concept", "label_language_status": status,
        })
        wd_label_rows += C.make_label_rows(eid, "occupation", WIKIDATA, preferred=pref, alts=alts)

    C.replace_source_rows(C.OCCUPATIONS_CSV, C.OCCUPATION_FIELDS, WIKIDATA, wd_occ_rows)
    C.replace_source_rows(C.LABELS_CSV,      C.LABEL_FIELDS,      WIKIDATA, wd_label_rows)
    C.log_provenance(WIKIDATA, [{
        "entity_id": "ALL_WIKIDATA", "source": WIKIDATA, "source_version": "live",
        "retrieved_at": C.now_iso(), "retrieval_method": "SPARQL query.wikidata.org",
        "notes": f"{len(wd_occ_rows)} concepts, dont "
                 f"{sum(1 for r in wd_occ_rows if r['pref_label_fr'])} avec libellé fr",
    }])
    n_fr = sum(1 for r in wd_occ_rows if r["pref_label_fr"])
    print(f"{len(wd_occ_rows)} concepts Wikidata ingérés ({n_fr} avec libellé français).")
    print("\nExemples :")
    for r in wd_occ_rows[:12]:
        lab = r["pref_label_fr"] or r["pref_label_en"]
        print(f"   {r['source_id']:10s} {lab}")
else:
    print("Aucune donnée Wikidata à ingérer (cellule précédente non aboutie).")
    print("Le reste du pipeline fonctionne sans Wikidata ; réexécutez quand l'accès est disponible.")

3 concepts Wikidata ingérés (1 avec libellé français).

Exemples :
   Q1644347   audiologiste
   Q111535889 chemometrician
   Q112978310 computer science teacher


## Vérifications

In [12]:
csv.field_size_limit(10_000_000)
def load(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

occs = load(C.OCCUPATIONS_CSV)
labels = load(C.LABELS_CSV)
print("Occupations par source :", dict(Counter(o["source"] for o in occs)))
print("Labels par source      :", dict(Counter(l["source"] for l in labels)))

# intégrité : tous les labels pointent vers une entité existante
occ_ids = {o["entity_id"] for o in occs}
sk_ids = {s["entity_id"] for s in load(C.SKILLS_CSV)}
dangling = [l for l in labels if l["entity_id"] not in (occ_ids | sk_ids)]
assert not dangling, f"{len(dangling)} labels pendants !"
print("\nIntégrité référentielle des labels : OK")

Occupations par source : {'ESCO': 83, 'ISCO': 20, 'REMOTEOK': 39, 'ARBEITNOW': 137, 'ROME': 96, 'WIKIDATA': 3}
Labels par source      : {'ESCO': 3531, 'ISCO': 20, 'REMOTEOK': 116, 'ARBEITNOW': 187, 'ROME': 2755, 'WIKIDATA': 12}

Intégrité référentielle des labels : OK


----

# 04 — Ingestion d'offres d'emploi (RemoteOK + Arbeitnow)

On va récupèrer des **offres d'emploi réelles** via **deux API JSON publiques et gratuites** :
- **RemoteOK** — `https://remoteok.com/api`
- **Arbeitnow** — `https://www.arbeitnow.com/api/job-board-api`

In [13]:
import re, json
import requests

RAW_DIR = os.path.join(C.ROOT, "scraped")
os.makedirs(RAW_DIR, exist_ok=True)

## 4.1. Paramètres

In [14]:
CONTACT_EMAIL = "my-email@exemple.com"
PAUSE_SECONDES = 1.5
MAX_PER_SOURCE = 300

USER_AGENT = (f"JobKB-research-project/1.0 (educational knowledge-graph project; "
              f"contact: {CONTACT_EMAIL})")

REMOTEOK_URL = "https://remoteok.com/api"
ARBEITNOW_URL = "https://www.arbeitnow.com/api/job-board-api"

def fetch_json(url, cache_name):
    cache = os.path.join(RAW_DIR, cache_name)

    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
        r.raise_for_status()
        data = r.json()
        with open(cache, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
        return data
    except Exception as e:
        print(f"  [réseau] échec {url} : {type(e).__name__} {str(e)[:100]}")
        
    if os.path.isfile(cache):
        print(f"  [cache] lecture de {cache_name}")
        with open(cache, encoding="utf-8") as f:
            return json.load(f)
    return None

print(f"Sources : RemoteOK + Arbeitnow — max {MAX_PER_SOURCE} offres/source.")

Sources : RemoteOK + Arbeitnow — max 300 offres/source.


## 4.2. Filtre « domaine IT »

In [15]:
IT_KEYWORDS = {
    "developer", "engineer", "programmer", "data", "software", "devops", "cloud",
    "machine learning", "ml", "ai", "artificial intelligence", "scientist", "analyst",
    "architect", "backend", "frontend", "full stack", "fullstack", "sre", "security",
    "cyber", "network", "database", "sysadmin", "qa", "test", "mobile", "web",
    "blockchain", "scrum", "product owner", "administrator", "it ",
    "python", "java", "javascript", "typescript", "react", "node", "kubernetes",
    "docker", "aws", "azure", "gcp", "sql", "nosql", "golang", "rust", "c++",
    "tensorflow", "pytorch", "spark", "kafka", "terraform",
}

def is_it_job(title, tags):
    hay = (title or "").lower() + " " + " ".join(t.lower() for t in (tags or []))
    return any(kw in hay for kw in IT_KEYWORDS)

def norm_tag(t):
    return re.sub(r"\s+", " ", str(t or "")).strip()

print(f"{len(IT_KEYWORDS)} mots-clés de filtrage IT.")

54 mots-clés de filtrage IT.


## 4.3. Récupération

In [16]:
def harvest_remoteok():
    data = fetch_json(REMOTEOK_URL, "remoteok.json")
    jobs = []
    if not isinstance(data, list):
        print("  RemoteOK : pas de données exploitables."); return jobs
    for item in data:
        if not isinstance(item, dict): continue
        title = norm_tag(item.get("position") or item.get("title") or "")
        if not title: continue
        tags = [norm_tag(t) for t in (item.get("tags") or []) if norm_tag(t)]
        if not is_it_job(title, tags): continue
        jobs.append({"title": title, "tags": tags,
                     "source_id": str(item.get("id") or item.get("slug") or title)})
    return jobs

def harvest_arbeitnow(max_pages=3):
    jobs = []
    for page in range(1, max_pages + 1):
        url = ARBEITNOW_URL if page == 1 else f"{ARBEITNOW_URL}?page={page}"
        data = fetch_json(url, f"arbeitnow_p{page}.json")
        rows = data.get("data") if isinstance(data, dict) else None
        if not rows: break
        for item in rows:
            if not isinstance(item, dict): continue
            title = norm_tag(item.get("title") or "")
            if not title: continue
            tags = [norm_tag(t) for t in (item.get("tags") or []) if norm_tag(t)]
            if not is_it_job(title, tags): continue
            jobs.append({"title": title, "tags": tags,
                         "source_id": str(item.get("slug") or title)})
        time.sleep(PAUSE_SECONDES)
    return jobs

remoteok_jobs = harvest_remoteok()[:MAX_PER_SOURCE]
print(f"RemoteOK : {len(remoteok_jobs)} offres IT retenues.")
time.sleep(PAUSE_SECONDES)
arbeitnow_jobs = harvest_arbeitnow()[:MAX_PER_SOURCE]
print(f"Arbeitnow : {len(arbeitnow_jobs)} offres IT retenues.")

RemoteOK : 40 offres IT retenues.
Arbeitnow : 145 offres IT retenues.


## 4.4. Écriture au format canonique

In [17]:
def ingest_source(jobs, SOURCE):
    usable = [j for j in jobs if j["tags"]]
    title_groups = defaultdict(list)
    for j in usable:
        title_groups[C.normalize_label(j["title"])].append(j)

    def occ_id(n): return C.mint_id("OCC_", SOURCE, n)
    def skl_id(n): return C.mint_id("SKL_", SOURCE, n)

    occ_rows, occ_label_rows, occ_skills = [], [], {}
    for norm, group in title_groups.items():
        best_label = Counter(j["title"] for j in group).most_common(1)[0][0]
        skills = set()
        for j in group:
            skills.update(j["tags"])
        occ_skills[norm] = skills
        eid = occ_id(norm)
        occ_rows.append({
            "entity_id": eid, "source": SOURCE, "source_id": norm, "isco_code": "",
            "pref_label_fr": "", "pref_label_en": best_label,
            "alt_labels_fr": "", "alt_labels_en": "",
            "description_fr": "", "description_en": "",
            "occupation_type": "job_posting_title",
            "label_language_status": "en_only_pending_fr",
        })
        occ_label_rows += C.make_label_rows(eid, "occupation", SOURCE,
                                            preferred={"en": [best_label]})

    skill_disp = {}
    for j in usable:
        for t in j["tags"]:
            n = C.normalize_label(t)
            if n: skill_disp.setdefault(n, t)

    skl_rows, skl_label_rows = [], []
    for norm, disp in skill_disp.items():
        eid = skl_id(norm)
        skl_rows.append({
            "entity_id": eid, "source": SOURCE, "source_id": norm,
            "pref_label_fr": "", "pref_label_en": disp,
            "alt_labels_fr": "", "alt_labels_en": "",
            "description_fr": "", "description_en": "",
            "esco_skill_type": "", "esco_reuse_level": "",
            "hard_soft_provisional": "hard", "hard_soft_method": "job_api_scraped",
        })
        skl_label_rows += C.make_label_rows(eid, "skill", SOURCE, preferred={"en": [disp]})

    rel_rows, seen = [], set()
    for norm, skills in occ_skills.items():
        oeid = occ_id(norm)
        for t in skills:
            ns = C.normalize_label(t)
            if not ns: continue
            seid = skl_id(ns)
            if (oeid, seid) in seen: continue
            seen.add((oeid, seid))
            rel_rows.append({"occupation_entity_id": oeid, "skill_entity_id": seid,
                             "relation_type": "essential", "source": SOURCE})

    C.replace_source_rows(C.OCCUPATIONS_CSV, C.OCCUPATION_FIELDS, SOURCE, occ_rows)
    C.replace_source_rows(C.SKILLS_CSV, C.SKILL_FIELDS, SOURCE, skl_rows)
    C.replace_source_rows(C.LABELS_CSV, C.LABEL_FIELDS, SOURCE,
                          occ_label_rows + skl_label_rows)
    C.replace_source_rows(C.OCC_SKILL_REL_CSV, C.REL_FIELDS, SOURCE, rel_rows)
    C.log_provenance(SOURCE, [{
        "entity_id": f"ALL_{SOURCE}", "source": SOURCE,
        "source_version": "API JSON publique", "retrieved_at": C.now_iso(),
        "retrieval_method": "API JSON officielle (sans clé)",
        "notes": f"{len(occ_rows)} intitulés, {len(skl_rows)} compétences, {len(rel_rows)} liens",
    }])
    return len(occ_rows), len(skl_rows), len(rel_rows)

for jobs, SOURCE in [(remoteok_jobs, "REMOTEOK"), (arbeitnow_jobs, "ARBEITNOW")]:
    o, s, r = ingest_source(jobs, SOURCE)
    print(f"{SOURCE:10s} -> {o} occupations, {s} compétences, {r} liens")

REMOTEOK   -> 39 occupations, 77 compétences, 676 liens
ARBEITNOW  -> 127 occupations, 50 compétences, 193 liens
